In [1]:
!pip install -q pdfplumber unicodeconverter


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pdfplumber
import pandas as pd
import unicodeconverter as uc

def decode_dghs_table(pdf_path, page_num=3): # Page 4 is index 3
    print("Extracting deterministic geometry...")
    with pdfplumber.open(pdf_path) as pdf:
        page = pdf.pages[page_num]
        
        # pdfplumber extracts the exact grid, but the text will be Bijoy/ANSI gibberish
        raw_table = page.extract_table()
        
    if not raw_table:
        print("No structural table found. Document might be a flat scanned image.")
        return pd.DataFrame()

    print("Decoding ANSI/Bijoy cipher to Unicode...")
    decoded_matrix = []
    
    for row in raw_table:
        decoded_row = []
        for cell in row:
            if cell:
                # Clean up bounding box artifacts and newline characters
                clean_cell = cell.replace('\n', ' ').strip()
                try:
                    # The magic happens here: translating the gibberish cipher
                    unicode_text = uc.convert_bijoy_to_unicode(clean_cell)
                    decoded_row.append(unicode_text)
                except Exception:
                    # Fallback if cell contains numbers or un-convertible characters
                    decoded_row.append(clean_cell)
            else:
                decoded_row.append("")
        decoded_matrix.append(decoded_row)

    # Convert the decoded matrix directly into a clean DataFrame
    df = pd.DataFrame(decoded_matrix[1:], columns=decoded_matrix[0])
    return df

# Execution


In [4]:
pdf_file = r"D:\ALL CODES\MATLAB PROJECT\PDF\20230304_dengue_all.pdf" 
clean_dataframe = decode_dghs_table(pdf_file)
clean_dataframe

Extracting deterministic geometry...
Decoding ANSI/Bijoy cipher to Unicode...


,,,,,,,,,,,০৪/০৩/২০২৩
0,রেভাগ,ক্রর ক নিং,বজলা,প্ররতষ্ঠামনি না,২৪ ঘন্টায় নত্যন ভরতব,,,গত ০১-০১-২০২৩ হমত অদ্যােরি,,,েত ব ামন ভরত ব বিাগী
1,,,,,সিকািী,বেসিকািী,ব াট,সেমব াট,মৃত্যু,ছাড়পত্র প্রাপ্ত বিাগী,
2,ঢাকা হানগি,,,,১,০,১,৩৫৭,৬,৩৩৯,১২
3,ঢাকা রেভাগ (ঢাকা হি ব্যতীত),১,ঢাকা,(ঢাকা হানগি ব্যারতত),০,০,০,৭,,৭,০
4,,২,ফরিদপুি,,০,০,০,০,,০,০
5,,৩,গাজীপুি,,০,০,০,১৭,,১৭,০
6,,৪,বগাপালগঞ্জ,,০,০,০,০,,০,০
7,,৫,রকম ািগঞ্জ,,০,০,০,৪,,৪,০
8,,৬,াদািীপুি,,০,০,০,১২,,১২,০
9,,৭,ারনকগঞ্জ,,০,০,০,৮,,৮,০


In [7]:
pdf_file = r"PDF\20250220_dengue_all.pdf" 
clean_dataframe2 = decode_dghs_table(pdf_file,page_num=8)
clean_dataframe2

Extracting deterministic geometry...
Decoding ANSI/Bijoy cipher to Unicode...


,স্বাস্থ্ু অরিদপ্তমিি বহলথ্ ই ামজিব ী অপামিশন বসন্টাি ও কমরাল রুম ঢাকা হানগিসহ রেরভন্ন রেভাগ বথমক বপ্ররিত বিঙ্গু বিামগ আক্রান্ত ও মৃত্যুি প্ররতমেদন । তারিখ:,,,,,,,,,,২০/০২/২০২৫
0,রেভাগ,ক্রর ক নিং,বজলা,প্ররতষ্ঠামনি না,২৪ ঘন্টায় নত্যন ভরতব,,,গত ০১-০১-২০২৫ হমত অদ্যােরি,,,েত ব ামন ভরত ববিাগী
1,,,,,সিকািী,বেসিকািী,ব াট,সেমব াট ভরতব,মৃত্যু,ছাড়পত্র প্রাপ্ত বিাগী,
2,,২৪,চট্টগ্রা,,০,০,০,২৮,০,২৭,১
3,,৩৫,চট্টগ্রা,চট্টগ্রা ব রিমকল কমলজ হাসপাতাল,০,০,০,৬১,১,৫৮,২
4,,চট্টগ্রা বজলাি ব াটেঃ,,,০,০,০,৮৯,১,৮৫,৩
5,চট্টগ্রা রেভাগ,২৫,কক্সোজাি,,০,০,০,৩৭,০,৩৬,১
6,,২৮,োন্দিোন,,০,০,০,১৪,০,১৩,১
7,,২৬,িাঙ্গা াটি,,০,০,০,৩,০,৩,০
8,,২৭,খাগড়াছরড়,,০,০,০,৯,০,৯,০
9,,২৯,বফনী,,০,০,০,২,০,২,০


In [13]:
import pdfplumber
import pandas as pd
import unicodeconverter as uc

def extract_full_dengue_table(pdf_path):
    all_raw_rows = []
    
    print("Scanning PDF pages for the target table...")
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            # 1. Extract raw text from the page (this will be the Bijoy gibberish)
            raw_text = page.extract_text()
            print(raw_text)
            
            if raw_text:
                # Clean newlines and convert the entire page's text to Unicode
                clean_text = raw_text.replace('\n', ' ')
                try:
                    unicode_text = uc.convert_bijoy_to_unicode(clean_text)
                except Exception:
                    unicode_text = ""
                
                # 2. Search for a robust, clean Unicode substring 
                # This ignores the random whitespace issues of the raw PDF layer
                if "হেলথ ইমার্জেন্সি অপারেশন সেন্টার" in unicode_text or "ডেঙ্গু রোগে আক্রান্ত" in unicode_text:
                    print(f"Target table found on Page {page.page_number}. Extracting matrix...")
                    
                    # 3. Extract the table from this specific page
                    table = page.extract_table()
                    if table:
                        all_raw_rows.extend(table)
                        
    if not all_raw_rows:
        print("Failure: Could not locate the table on any page.")
        return pd.DataFrame()

    print("Decoding aggregated multi-page matrix to Unicode...")
    decoded_matrix = []
    
    for row in all_raw_rows:
        decoded_row = []
        for cell in row:
            if cell:
                # Clean up bounding box artifacts
                clean_cell = cell.replace('\n', ' ').strip()
                try:
                    decoded_row.append(uc.convert_bijoy_to_unicode(clean_cell))
                except Exception:
                    decoded_row.append(clean_cell)
            else:
                decoded_row.append("")
        decoded_matrix.append(decoded_row)

    # 4. Handle multi-page header duplication
    # When tables span multiple pages, the PDF usually reprints the header row.
    # We grab the first row as the absolute header, and filter it out from the rest of the data.
    header = decoded_matrix[0]
    
    # Keep rows where the first column isn't repeating the word "বিভাগ"
    clean_data = [row for row in decoded_matrix[1:] if row and row[0] != "বিভাগ"]

    # 5. Build the final DataFrame
    df = pd.DataFrame(clean_data, columns=header)
    return df

# Execution
pdf_file = r"PDF\20230304_dengue_all.pdf" 
final_df   = extract_full_dengue_table(pdf_file)

# Display the total rows extracted and the first few lines
print(f"Successfully extracted {len(final_df)} rows across all pages.")


Scanning PDF pages using raw string matching...


FileNotFoundError: [Errno 2] No such file or directory: 'dghs_daily_dengue_report.pdf'

In [14]:
import pdfplumber
import pandas as pd
import unicodeconverter as uc

def extract_dengue_table_raw_match(pdf_path):
    all_raw_rows = []
    
    # Using exact substrings from your raw output, ignoring the date
    target_string_1 = "বিঙ্গু বিামগ আক্রান্ত ও মৃত্যুি প্ররতমেদন"
    target_string_2 = "বহলথ্ ই ামজিব ী অপামি ন বসন্টাি"
    
    print("Scanning PDF pages using raw string matching...")
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            raw_text = page.extract_text()
            
            if raw_text:
                # Direct string match against the exact raw PDF parser output
                if target_string_1 in raw_text or target_string_2 in raw_text:
                    print(f"Match found on Page {page.page_number}. Extracting matrix...")
                    
                    table = page.extract_table()
                    if table:
                        all_raw_rows.extend(table)
                        
    if not all_raw_rows:
        print("Failure: Could not locate the target raw string on any page.")
        return pd.DataFrame()

    print("Decoding aggregated matrix to Unicode...")
    decoded_matrix = []
    
    for row in all_raw_rows:
        decoded_row = []
        for cell in row:
            if cell:
                # Clean bounding box artifacts
                clean_cell = str(cell).replace('\n', ' ').strip()
                try:
                    # Translate the matrix contents to proper Bengali Unicode
                    decoded_row.append(uc.convert_bijoy_to_unicode(clean_cell))
                except Exception:
                    decoded_row.append(clean_cell)
            else:
                decoded_row.append("")
        decoded_matrix.append(decoded_row)

    # Clean up repeating headers from page breaks
    header = decoded_matrix[0]
    
    # Keep row if it's not empty and the first column isn't repeating "বিভাগ"
    clean_data = [
        row for row in decoded_matrix[1:] 
        if any(row) and "বিভাগ" not in str(row[0])
    ]

    df = pd.DataFrame(clean_data, columns=header)
    return df

# Execution
pdf_file = r"PDF\20230304_dengue_all.pdf" 
final_df = extract_dengue_table_raw_match(pdf_file)

print(f"Successfully extracted {len(final_df)} rows.")
print(final_df.head())

Scanning PDF pages using raw string matching...
Match found on Page 4. Extracting matrix...
Match found on Page 5. Extracting matrix...
Decoding aggregated matrix to Unicode...
Successfully extracted 87 rows.
                                                                           \
0                        রেভাগ  ক্রর ক নিং     বজলা       প্ররতষ্ঠামনি না   
1                                                                           
2                   ঢাকা হানগি                                              
3  ঢাকা রেভাগ (ঢাকা হি ব্যতীত)           ১     ঢাকা  (ঢাকা হানগি ব্যারতত)   
4                                        ২  ফরিদপুি                         

                                                                             \
0  ২৪ ঘন্টায় নত্যন ভরতব                  গত ০১-০১-২০২৩ হমত অদ্যােরি           
1                সিকািী  বেসিকািী  ব াট                     সেমব াট  মৃত্যু   
2                     ১         ০     ১                         ৩৫৭       ৬   
3           

,,,,,,,,,,,০৪/০৩/২০২৩
0,রেভাগ,ক্রর ক নিং,বজলা,প্ররতষ্ঠামনি না,২৪ ঘন্টায় নত্যন ভরতব,,,গত ০১-০১-২০২৩ হমত অদ্যােরি,,,েত ব ামন ভরত ব বিাগী
1,,,,,সিকািী,বেসিকািী,ব াট,সেমব াট,মৃত্যু,ছাড়পত্র প্রাপ্ত বিাগী,
2,ঢাকা হানগি,,,,১,০,১,৩৫৭,৬,৩৩৯,১২
3,ঢাকা রেভাগ (ঢাকা হি ব্যতীত),১,ঢাকা,(ঢাকা হানগি ব্যারতত),০,০,০,৭,,৭,০
4,,২,ফরিদপুি,,০,০,০,০,,০,০
...,...,...,...,...,...,...,...,...,...,...,...
82,,৭১,ব ৌলভীোজাি,,০,০,০,০,,০,০
83,,৭২,রসমলট,এ .এ.রজ ওস ারন ব েঃ কেঃ হােঃ,০,০,০,০,,০,০
84,,,,,০,০,০,৩,০,৩,০
85,রেভামগি সেমব াট,,,,১,০,১,৩৯৭,৩,৩৭৮,১৬


In [9]:
print(final_df.head())

Empty DataFrame
Columns: []
Index: []


In [27]:
import pdfplumber
import pandas as pd
import unicodeconverter as uc
import re

def extract_dengue_table_robust(pdf_path):
    all_raw_rows = []
    
    # Define the "Search signatures" for the target page
    # 1. The literal raw gibberish from your PDF parser
    raw_signature = "বিঙ্গু বিামগ আক্রান্ত ও মৃত্যুি প্ররতমেদন"
    # 2. The clean Unicode version
    unicode_signature = "ডেঙ্গু রোগে আক্রান্ত ও মৃত্যুর প্রতিবেদন"
    
    print("Initiating multi-strategy scan...")
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            raw_text = page.extract_text()
            if not raw_text:
                continue

            # Normalized text (removing all spaces/newlines) to catch kerning variations
            normalized_raw = re.sub(r'\s+', '', raw_text)
            
            # 3. Try to convert the whole page to Unicode for searching
            try:
                unicode_page_text = uc.convert_bijoy_to_unicode(raw_text)
            except:
                unicode_page_text = ""

            # MULTI-STRATEGY CHECK: If any of these hit, we found the page
            found = (
                raw_signature in raw_text or 
                unicode_signature in unicode_page_text or
                re.sub(r'\s+', '', raw_signature) in normalized_raw
            )

            if found:
                print(f"Table found on Page {page.page_number} via signature match.")
                table = page.extract_table()
                if table:
                    # Filter out any 'None' rows that might appear in the grid
                    table = [r for r in table if r is not None]
                    all_raw_rows.extend(table)

    if not all_raw_rows:
        return pd.DataFrame()

    print("Decoding matrix cells...")
    decoded_matrix = []
    for row in all_raw_rows:
        decoded_row = []
        for cell in row:
            if cell:
                # Standardize cell text by removing problematic newlines
                clean_cell = str(cell).replace('\n', ' ').strip()
                try:
                    decoded_row.append(uc.convert_bijoy_to_unicode(clean_cell))
                except:
                    decoded_row.append(clean_cell)
            else:
                decoded_row.append("")
        decoded_matrix.append(decoded_row)

    # Reconstruct DataFrame and drop repeated headers from page breaks
    header = decoded_matrix[0]
    # Keep rows that aren't headers and aren't purely empty
    final_rows = [r for r in decoded_matrix[1:] if any(r) and "বিভাগ" not in str(r[0])]
    
    return pd.DataFrame(final_rows, columns=header)

# Execution
pdf_file = r"PDF\20250220_dengue_all.pdf" 
df = extract_dengue_table_robust(pdf_file)

# Preview


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Initiating multi-strategy scan...


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Table found on Page 8 via signature match.
Table found on Page 9 via signature match.
Table found on Page 10 via signature match.
Decoding matrix cells...


In [28]:
df.head()

,"গণপ্রজাতন্ত্রী োিংলামদশ সিকাি স্বাস্থ্ু ও পরিোি কল্যাণ ন্ত্রণালয় স্বাস্থ্ু বসো রেভাগ স্বাস্থ্ু অরিদপ্তি, এ আই এস বহলথ ই ামজিব ী অপামিশন বসন্টাি ও কমরাল রু হাখালী, ঢাকা-১২১২",,,,,,,,,,
0,স্বাস্থ্ু অরিদপ্তমিি বহলথ্ ই ামজিব ী অপামিশন ব...,,,,,,,,,,২০/০২/২০২৫
1,রেভাগ,ক্রর ক নিং,বজলা,প্ররতষ্ঠামনি না,২৪ ঘন্টায় নত্যন ভরতব,,,গত ০১-০১-২০২৫ হমত অদ্যােরি,,,েত ব ামন ভরত ববিাগী
2,,,,,সিকািী,বেসিকািী,ব াট,সেমব াট ভরতব,মৃত্যু,ছাড়পত্র প্রাপ্ত বিাগী,
3,ঢাকা হানগি,,,,১,০,১,৫০০,৯,৪৫৫,৩৬
4,,১,ঢাকা,(ঢাকা হানগি ব্যারতত),০,০,০,১৪,০,১৩,১


In [29]:
# python
df.to_csv("HELLO2.csv"  , index=False) #, orient="records")

In [31]:
!pip install tabulate


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [32]:
df.to_markdown("HELLO2.md", index=False)

In [41]:
import pandas as pd
import csv

def process_broken_dengue_csv(filepath):
    # 1. Custom Translation Dictionaries (Gibberish -> Clean Unicode)
    # You can expand this dictionary as you find more districts in your dataset
    string_map = {
        "ঢাকা হানগি": "ঢাকা মহানগর",
        "ঢাকা রেভাগ (ঢাকা হি ব্যতীত)": "ঢাকা বিভাগ",
        "ঢাকা": "ঢাকা",
        "ফরিদপুি": "ফরিদপুর",
        "গাজীপুি": "গাজীপুর",
        "বগাপালগঞ্জ": "গোপালগঞ্জ",
        "রকম ািগঞ্জ": "কিশোরগঞ্জ",
        "াদািীপুি": "মাদারীপুর",
        "ারনকগঞ্জ": "মানিকগঞ্জ",
        "মুরিরগঞ্জ": "মুন্সিগঞ্জ",
        "নািায়ণগঞ্জ": "নারায়ণগঞ্জ",
        "নিরসিংদী": "নরসিংদী",
        "িাজোড়ী": "রাজবাড়ী",
        "িাজ াহী রেভাগ": "রাজশাহী বিভাগ",
        "িাজ াহী": "রাজশাহী",
        "চাপাইনোেগঞ্জ": "চাঁপাইনবাবগঞ্জ",
        "নওগাঁ": "নওগাঁ",
        "নামটাি": "নাটোর",
        "জয়পুিহাট": "জয়পুরহাট",
        "েগুড়া": "বগুড়া",
        "রসিাজগঞ্জ": "সিরাজগঞ্জ",
        "পােনা": "পাবনা",
        "িিংপুি রেভাগ": "রংপুর বিভাগ",
        "িিংপুি": "রংপুর",
        "লাল রনিহাট": "লালমনিরহাট",
        "কুরড়গ্রা": "কুড়িগ্রাম",
        "নীলফা ািী": "নীলফামারী",
        "রদনাজপুি": "দিনাজপুর",
        "গাইোন্ধা": "গাইবান্ধা",
        "ঠাকুিগাঁও": "ঠাকুরগাঁও",
        "পঞ্চগড়": "পঞ্চগড়"
    }

    # Bengali to English digit mapping
    digit_map = str.maketrans('০১২৩৪৫৬৭৮৯', '0123456789')

    extracted_data = []
    
    print(f"Processing {filepath}...")
    
    # 2. Read the CSV manually to avoid Pandas header inference errors
    with open(filepath, 'r', encoding='utf-8-sig') as file:
        reader = csv.reader(file)
        rows = list(reader)

    header_idx = -1
    
    # 3. Geometrically scan for the exact row where the table starts
    for i, row in enumerate(rows):
        # Look for the row containing "রেভাগ" (Division) and "বজলা" (District)
        if len(row) > 2 and "রেভাগ" in row[0] and "বজলা" in row[2]:
            header_idx = i
            break
            
    if header_idx == -1:
        print("Error: Could not find table headers in this file.")
        return pd.DataFrame()

    # The data effectively starts 2 rows down (skipping the sub-columns row like সরকারী, বেসরকারী, ব াট)
    data_rows = rows[header_idx + 2:]
    
    current_division = "Unknown"
    
    # 4. Process the rows
    for row in data_rows:
        # Stop processing if we hit empty rows or summary rows at the bottom
        if not row or len(row) < 7:
            continue
            
        raw_div = row[0].strip()
        raw_dist = row[2].strip()
        
        # Forward-fill the Division. If the column isn't empty, update the current division.
        if raw_div:
            current_division = raw_div
            
        # Handle special case: Dhaka City often doesn't list a district, it just lists the division
        if current_division == "ঢাকা হানগি" and not raw_dist:
            raw_dist = "ঢাকা হানগি"
            
        # Filter out subtotals (e.g., "ফরিদপুি বজলাি ব াটেঃ" -> Faridpur District Total)
        if "ব াটেঃ" in raw_div or "ব াটেঃ" in raw_dist or "সেমব াট" in raw_div:
            continue
            
        # 5. Extract the 24 Hour Cases (Total). In both your CSVs, this is at index 6.
        raw_cases_24h = row[6].strip()
        
        # Convert Bengali numerals to standard integers (default to 0 if blank)
        cases_24h_eng = raw_cases_24h.translate(digit_map)
        try:
            cases_integer = int(cases_24h_eng)
        except ValueError:
            cases_integer = 0
            
        # Only keep rows that actually have a district mentioned
        if raw_dist:
            # 6. Apply string mappings to clean the text
            clean_division = string_map.get(current_division, current_division)
            clean_district = string_map.get(raw_dist, raw_dist)
            
            extracted_data.append({
                "Division": clean_division,
                "District": clean_district,
                "Cases_in_24_Hours": cases_integer
            })

    # Convert to a clean Pandas DataFrame
    df = pd.DataFrame(extracted_data)
    
    return df

# Example Usage:
df_2023 = process_broken_dengue_csv('HELLO.csv')
# df_2025 = process_broken_dengue_csv('HELLO2.csv')
# combined_df = pd.concat([df_2023, df_2025], ignore_index=True)
# print(combined_df.head(20))

--- Extracted Clean English Data ---
| Division            | District    |   Cases_in_24_Hours |
|:--------------------|:------------|--------------------:|
| Dhaka City          | Dhaka City  |                   1 |
| Dhaka Division      | Dhaka       |                   0 |
| Dhaka Division      | Faridpur    |                   0 |
| Dhaka Division      | Gazipur     |                   0 |
| Dhaka Division      | Gopalganj   |                   0 |
| Dhaka Division      | Kishoreganj |                   0 |
| Dhaka Division      | Madaripur   |                   0 |
| Dhaka Division      | Manikganj   |                   0 |
| Dhaka Division      | Munshiganj  |                   0 |
| Dhaka Division      | Narayanganj |                   0 |
| Dhaka Division      | Narsingdi   |                   0 |
| Dhaka Division      | Rajbari     |                   0 |
| Dhaka Division      | Shariatpur  |                   0 |
| Dhaka Division      | Tangail     |                   0 |
| M

In [47]:
import pandas as pd
import csv

def extract_english_dengue_dataset(filepath):
    # 1. Direct Translation Dictionary: Bijoy Gibberish -> Pure English
    english_map = {
        # Divisions / Special Regions
        "ঢাকা হানগি": "Dhaka City",
        "ঢাকা রেভাগ (ঢাকা হি ব্যতীত)": "Dhaka Division",
        "িাজ াহী রেভাগ": "Rajshahi Division",
        "িিংপুি রেভাগ": "Rangpur Division",
        "খুলনা রেভাগ": "Khulna Division",
        "চট্রগ্রা  রেভাগ": "Chittagong Division",
        "েরিশাল রেভাগ": "Barisal Division",
        "রসমলট রেভাগ": "Sylhet Division",
        "য় নরসিংহ রেভাগ": "Mymensingh Division",
        
        # Districts
        "ঢাকা": "Dhaka",
        "ফরিদপুি": "Faridpur",
        "গাজীপুি": "Gazipur",
        "বগাপালগঞ্জ": "Gopalganj",
        "রকম ািগঞ্জ": "Kishoreganj",
        "াদািীপুি": "Madaripur",
        "ারনকগঞ্জ": "Manikganj",
        "মুরিরগঞ্জ": "Munshiganj",
        "নািায়ণগঞ্জ": "Narayanganj",
        "নিরসিংদী": "Narsingdi",
        "িাজোড়ী": "Rajbari",
        "িাজ াহী": "Rajshahi",
        "চাপাইনোেগঞ্জ": "Chapainawabganj",
        "নওগাঁ": "Naogaon",
        "নামটাি": "Natore",
        "জয়পুিহাট": "Joypurhat",
        "েগুড়া": "Bogra",
        "রসিাজগঞ্জ": "Sirajganj",
        "পােনা": "Pabna",
        "িিংপুি": "Rangpur",
        "লাল রনিহাট": "Lalmonirhat",
        "কুরড়গ্রা": "Kurigram",
        "নীলফা ািী": "Nilphamari",
        "রদনাজপুি": "Dinajpur",
        "গাইোন্ধা": "Gaibandha",
        "ঠাকুিগাঁও": "Thakurgaon",
        "পঞ্চগড়": "Panchagarh"
        # Note: You can easily add the remaining districts to this dictionary if they appear
    }

    # Bengali to English digit mapping
    digit_map = str.maketrans('০১২৩৪৫৬৭৮৯', '0123456789')

    extracted_data = []
    
    print(f"Scanning {filepath} for table boundaries...")
    
    with open(filepath, 'r', encoding='utf-8-sig') as file:
        reader = csv.reader(file)
        rows = list(reader)

    header_idx = -1
    
    # Geometrically scan for the exact row where the table starts
    for i, row in enumerate(rows):
        if len(row) > 2 and "রেভাগ" in row[0] and "বজলা" in row[2]:
            header_idx = i
            break
            
    if header_idx == -1:
        print("Error: Could not find table headers.")
        return pd.DataFrame()

    # The actual numerical data starts 2 rows down
    data_rows = rows[header_idx + 2:]
    
    current_division = "Unknown"
    
    for row in data_rows:
        if not row or len(row) < 7:
            continue
            
        raw_div = row[0].strip()
        raw_dist = row[2].strip()
        
        # Forward-fill the Division column
        if raw_div:
            current_division = raw_div
            
        # Dhaka City edge case: It often leaves the district blank in these reports
        if current_division == "ঢাকা হানগি" and not raw_dist:
            raw_dist = "ঢাকা হানগি"
            
        # Strip out the aggregate/subtotal rows (e.g., Faridpur Total)
        if "ব াটেঃ" in raw_div or "ব াটেঃ" in raw_dist or "সেমব াট" in raw_div:
            continue
            
        # Extract 24 Hour Total Cases (Index 6)
        raw_cases_24h = row[6].strip()
        
        # Convert Bengali string digits to standard integers
        cases_24h_eng = raw_cases_24h.translate(digit_map)
        try:
            cases_integer = int(cases_24h_eng)
        except ValueError:
            cases_integer = 0
            
        if raw_dist:
            # Map the gibberish directly to English, fallback to raw if not found in dict
            english_division = english_map.get(current_division, current_division)
            english_district = english_map.get(raw_dist, raw_dist)
            
            # Construct the clean, fully English row
            extracted_data.append({
                "Division": english_division,
                "District": english_district,
                "Cases_in_24_Hours": cases_integer
            })

    df = pd.DataFrame(extracted_data)
    return df

# Execution
df_2023 = extract_english_dengue_dataset('HELLO2.csv')
df_2023.to_csv("Dengue_Cases_2025.csv", index=False)

Scanning HELLO2.csv for table boundaries...


In [48]:
import pandas as pd

def clean_dengue_csv_to_english(filepath):
    # 1. The Comprehensive Bijoy-to-English Master Dictionary
    # This accounts for EVERY unique variation found in your 2023/2025 data.
    english_map = {
        # --- Divisions ---
        "ঢাকা হানগি": "Dhaka City",
        "ঢাকা রেভাগ (ঢাকা হি ব্যতীত)": "Dhaka Division",
        "িাজ াহী রেভাগ": "Rajshahi Division",
        "িিংপুি রেভাগ": "Rangpur Division",
        "খুলনা রেভাগ": "Khulna Division",
        "চট্টগ্রা রেভাগ": "Chittagong Division",
        "চট্রগ্রা  রেভাগ": "Chittagong Division", # Accounts for double-space variation
        "েরি াল রেভাগ": "Barisal Division",
        "েরিশাল রেভাগ": "Barisal Division", # Spelling variation
        "রসমলট রেভাগ": "Sylhet Division",
        "য় নরসিংহ রেভাগ": "Mymensingh Division",
        
        # --- Districts (Dhaka Division) ---
        "ঢাকা": "Dhaka",
        "ফরিদপুি": "Faridpur",
        "গাজীপুি": "Gazipur",
        "বগাপালগঞ্জ": "Gopalganj",
        "রকম ািগঞ্জ": "Kishoreganj",
        "াদািীপুি": "Madaripur",
        "ারনকগঞ্জ": "Manikganj",
        "মুরিরগঞ্জ": "Munshiganj",
        "নািায়ণগঞ্জ": "Narayanganj",
        "নিরসিংদী": "Narsingdi",
        "িাজোড়ী": "Rajbari",
        "িীয়তপুি": "Shariatpur", # Missing in previous iterations
        "টাঙ্গাইল": "Tangail",     # Missing in previous iterations
        
        # --- Districts (Mymensingh Division) ---
        "য় নরসিংহ": "Mymensingh",
        "জা ালপুি": "Jamalpur",
        "ব িপুি": "Sherpur",
        "বনত্রমকানা": "Netrokona",
        
        # --- Districts (Chittagong Division) ---
        "চট্টগ্রা": "Chittagong",
        "কক্সোজাি": "Cox's Bazar",
        "িাঙ্গা াটি": "Rangamati",
        "খাগড়াছরড়": "Khagrachari",
        "োন্দিোন": "Bandarban",
        "বফনী": "Feni",
        "কুর ল্লা": "Comilla",
        "চাঁদপুি": "Chandpur",
        "লক্ষীপুি": "Lakshmipur",
        "বনায়াখালী": "Noakhali",
        "ব্রাহ্মনোরড়য়া": "Brahmanbaria",
        
        # --- Districts (Khulna Division) ---
        "খুলনা": "Khulna",
        "োমগিহাট": "Bagerhat",
        "সাতক্ষীিা": "Satkhira",
        "যম াি": "Jessore",
        "রিনাইদহ": "Jhenaidah",
        "াগুিা": "Magura",
        "নড়াইল": "Narail",
        "কুরিয়া": "Kushtia",
        "চুয়ািাঙ্গা": "Chuadanga",
        "ব মহিপুি": "Meherpur",
        
        # --- Districts (Rajshahi Division) ---
        "িাজ াহী": "Rajshahi",
        "চাপাইনোেগঞ্জ": "Chapainawabganj",
        "নওগাঁ": "Naogaon",
        "নামটাি": "Natore",
        "জয়পুিহাট": "Joypurhat",
        "েগুড়া": "Bogra",
        "রসিাজগঞ্জ": "Sirajganj",
        "পােনা": "Pabna",
        
        # --- Districts (Rangpur Division) ---
        "িিংপুি": "Rangpur",
        "লাল রনিহাট": "Lalmonirhat",
        "কুরড়গ্রা": "Kurigram",
        "নীলফা ািী": "Nilphamari",
        "রদনাজপুি": "Dinajpur",
        "গাইোন্ধা": "Gaibandha",
        "ঠাকুিগাঁও": "Thakurgaon",
        "পঞ্চগড়": "Panchagarh",
        
        # --- Districts (Barisal Division) ---
        "েরি াল": "Barisal",
        "পটুয়াখালী": "Patuakhali",
        "বভালা": "Bhola",
        "রপমিাজপুি": "Pirojpur",
        "েিগুনা": "Barguna",
        "িালকাঠি": "Jhalokati",
        
        # --- Districts (Sylhet Division) ---
        "রসমলট": "Sylhet",
        "সুনা গঞ্জ": "Sunamganj",
        "হরেগঞ্জ": "Habiganj",
        "ব ৌলভীোজাি": "Moulvibazar"
    }

    # 2. Load the semi-broken CSV you already extracted
    df = pd.read_csv(filepath)

    # 3. Clean string whitespace so the dictionary maps correctly
    df['Division'] = df['Division'].astype(str).str.strip()
    df['District'] = df['District'].astype(str).str.strip()

    # 4. Map everything to pure English using the dictionary
    # If a string isn't found in the dictionary, it keeps its current value (e.g., if it's already in English)
    df['Division'] = df['Division'].map(lambda x: english_map.get(x, x))
    df['District'] = df['District'].map(lambda x: english_map.get(x, x))

    # 5. ROBUSTNESS FILTERING: Remove stray page breaks and subtotals
    # Sometimes PDF parsers copy the table header "রেভাগ" (Division) into a row during a page break
    df = df[df['District'] != 'বজলা']
    df = df[df['Division'] != 'রেভাগ']
    
    # Remove aggregate Subtotal rows if they crept in (e.g., "ফরিদপুি বজলাি ব াটেঃ")
    df = df[~df['District'].str.contains('ব াটেঃ', na=False)]
    
    # Ensure all numerical values are actual integers, replacing anything weird with 0
    df['Cases_in_24_Hours'] = pd.to_numeric(df['Cases_in_24_Hours'], errors='coerce').fillna(0).astype(int)

    return df

# Execution
clean_df = clean_dengue_csv_to_english('Dengue_Cases_2025.csv')

# Verify the output
print("--- Extracted Clean English Data ---")
print(clean_df.head(20).to_markdown(index=False))

--- Extracted Clean English Data ---
| Division     | District    |   Cases_in_24_Hours |
|:-------------|:------------|--------------------:|
| Dhaka City   | Dhaka City  |                   1 |
| Dhaka City   | Dhaka       |                   0 |
| Dhaka City   | Faridpur    |                   0 |
| Dhaka City   | Faridpur    |                   0 |
| Dhaka City   | Dhaka City  |                   0 |
| Dhaka City   | Gazipur     |                   0 |
| Dhaka City   | Gazipur     |                   0 |
| Dhaka City   | Dhaka City  |                   0 |
| Dhaka City   | Gopalganj   |                   0 |
| Dhaka City   | রকমশািগঞ্জ     |                   0 |
| Dhaka City   | রকমশািগঞ্জ     |                   0 |
| (ঢাকা শহি ব্যতীত) | Madaripur   |                   0 |
| (ঢাকা শহি ব্যতীত) | Manikganj   |                   0 |
| (ঢাকা শহি ব্যতীত) | Manikganj   |                   0 |
| (ঢাকা শহি ব্যতীত) | মুরিগঞ্জ       |                   2 |
| (ঢাকা শহি ব্যতীত) | Narayanganj

In [49]:
clean_df.to_csv("Dengue_Cases_2025_Cleaned.csv", index=False)

In [37]:
df_2023.to_csv("Dengue_Cases_2023.csv", index=False)

In [24]:
# python
# detect duplicates
print(df.columns[df.columns.duplicated()].unique())

def make_unique(cols):
    seen = {}
    out = []
    for c in cols:
        if c in seen:
            seen[c] += 1
            out.append(f"{c}.{seen[c]}")
        else:
            seen[c] = 0
            out.append(c)
    return out

df.columns = make_unique(df.columns)
df.to_json("HELLO.json", orient="columns")

Index([''], dtype='object')
